# 🧹 Preprocesamiento — Detección de toxicidad en YouTube (`IsToxic`)

## 🎯 Objetivo
Pipeline completo de limpieza, lematización, split y data augmentation sobre
`youtoxic_english_1000.csv` con variable objetivo `IsToxic`.

## ⚠️ Decisión clave: split ANTES del augmentation
El augmentation se aplica **únicamente sobre el conjunto de entrenamiento** tras el split.
Si se hace antes, versiones sintéticas de textos del train pueden acabar en test →
métricas infladas → evaluación inválida en producción.

## 📋 Flujo del notebook
1. Setup e imports
2. Carga y selección de columnas
3. Limpieza con regex
4. Lematización con spaCy
5. Split estratificado train / val / test
6. Data Augmentation solo sobre train
7. Exportación de CSVs a `data/processed/V_04`

In [3]:
import re
import os
import time
import warnings

import pandas as pd
import numpy as np
import nltk
import spacy
from tqdm import tqdm
from deep_translator import GoogleTranslator
import nlpaug.augmenter.word as naw
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Recursos NLTK necesarios
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet',
                 'omw-1.4', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

# Crear carpeta de salida si no existe
os.makedirs('../data/processed/V_04', exist_ok=True)

TARGET       = 'IsToxic'
RANDOM_STATE = 42
AUG_THRESHOLD = 0.48  # ratio mínimo de clase tóxica en train antes de augmentar

print('Setup completado.')
print(f'Carpeta de salida: ../data/processed/V_04')

Setup completado.
Carpeta de salida: ../data/processed/V_04
